# 01. 문서 요약 Tool — PDF 전처리 + `summary.py`
**Day 4**

> 수업용 문서: **`samples/Language_Models.pdf`**
> (언어 모델 개요 자료 — 긴 PDF 요약 실습에 적합)

**파이프라인:** `PDF → PyMuPDF 추출 → 전처리 → OpenAI 요약 → 저장`

---
## 0. 설치

In [1]:
!pip install pymupdf openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 32.0 MB/s  0:00:00 eta 0:00:01


In [2]:
!pip install pymupdf

In [20]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
os.getcwd()


'/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차'

In [8]:
pdf_path = os.path.join(os.getcwd(),"samples/Language_models.pdf")

---
## 1. PyMuPDF로 `Language_Models.pdf` 텍스트 추출

In [9]:
import pymupdf

doc = pymupdf.open(pdf_path)

In [10]:
doc

Document('/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/Language_models.pdf')

In [11]:
i=0
for page in doc:
    text = page.get_text()
    print(text)
    i+=1
    if i==5:
        break

Language Models are Few-Shot Learners
Tom B. Brown∗
Benjamin Mann∗
Nick Ryder∗
Melanie Subbiah∗
Jared Kaplan†
Prafulla Dhariwal
Arvind Neelakantan
Pranav Shyam
Girish Sastry
Amanda Askell
Sandhini Agarwal
Ariel Herbert-Voss
Gretchen Krueger
Tom Henighan
Rewon Child
Aditya Ramesh
Daniel M. Ziegler
Jeffrey Wu
Clemens Winter
Christopher Hesse
Mark Chen
Eric Sigler
Mateusz Litwin
Scott Gray
Benjamin Chess
Jack Clark
Christopher Berner
Sam McCandlish
Alec Radford
Ilya Sutskever
Dario Amodei
OpenAI
Abstract
Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training
on a large corpus of text followed by ﬁne-tuning on a speciﬁc task. While typically task-agnostic
in architecture, this method still requires task-speciﬁc ﬁne-tuning datasets of thousands or tens of
thousands of examples. By contrast, humans can generally perform a new language task from only
a few examples or from simple instructions – something which current NLP systems still largely
struggle

In [13]:
full_text=''
for page in doc:
    text = page.get_text()
    full_text+=text + '\n------------------------\n'

full_text

'Language Models are Few-Shot Learners\nTom B. Brown∗\nBenjamin Mann∗\nNick Ryder∗\nMelanie Subbiah∗\nJared Kaplan†\nPrafulla Dhariwal\nArvind Neelakantan\nPranav Shyam\nGirish Sastry\nAmanda Askell\nSandhini Agarwal\nAriel Herbert-Voss\nGretchen Krueger\nTom Henighan\nRewon Child\nAditya Ramesh\nDaniel M. Ziegler\nJeffrey Wu\nClemens Winter\nChristopher Hesse\nMark Chen\nEric Sigler\nMateusz Litwin\nScott Gray\nBenjamin Chess\nJack Clark\nChristopher Berner\nSam McCandlish\nAlec Radford\nIlya Sutskever\nDario Amodei\nOpenAI\nAbstract\nRecent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training\non a large corpus of text followed by ﬁne-tuning on a speciﬁc task. While typically task-agnostic\nin architecture, this method still requires task-speciﬁc ﬁne-tuning datasets of thousands or tens of\nthousands of examples. By contrast, humans can generally perform a new language task from only\na few examples or from simple instructions – something which cur

In [14]:
pdf_file_name = os.path.basename(pdf_path)
pdf_file_name

'Language_models.pdf'

In [15]:
pdf_file_name = os.path.splitext(pdf_file_name)[0]
txt_file_path = os.path.join(os.getcwd(),'samples',f"{pdf_file_name}.txt")

In [16]:
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write(full_text)

---
## 2. 문서 요약

In [17]:
load_dotenv("../../.env")

api_key = os.getenv("OPENAI_KEY")

def summarize_txt(file_path):
    client = OpenAI(api_key=api_key)
    with open(file_path, 'r', encoding = 'utf-8') as f:
        txt = f.read()

    system_prompt = f'''
    너는 다음 글을 요약하는 봇이다. 아래 글을 읽고, 

    작성해야 하는 포맷은 다음과 같음
    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개


    ============= 이하 텍스트 ================
    {txt[:10000]}

    '''

    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        temperature = 0.1,
        messages=[
            {"role":"system","content":system_prompt},
        ]
    )

    return response.choices[0].message.content


In [18]:
summary = summarize_txt(txt_file_path)

In [19]:
print(summary)

# GPT-3: Few-Shot Learning in Language Models

## 저자의 문제 인식 및 주장
저자들은 최근 NLP(자연어 처리) 작업에서 대규모 텍스트 코퍼스를 사전 훈련한 후 특정 작업에 대해 미세 조정하는 방법이 상당한 성과를 거두었다고 주장한다. 그러나 이 방법은 여전히 수천 또는 수만 개의 예제가 필요한 작업별 데이터셋을 요구한다는 한계가 있다. 반면, 인간은 몇 가지 예제나 간단한 지시만으로도 새로운 언어 작업을 수행할 수 있다. 저자들은 언어 모델을 확장하면 작업에 구애받지 않는 몇 가지 샷 성능이 크게 향상된다고 보여준다. 특히, 1750억 개의 매개변수를 가진 GPT-3를 훈련하여 몇 가지 샷 설정에서 성능을 테스트하였다. GPT-3는 미세 조정 없이도 다양한 NLP 데이터셋에서 강력한 성능을 발휘하며, 즉석에서 추론이나 도메인 적응이 필요한 작업에서도 좋은 결과를 보인다. 그러나 저자들은 GPT-3가 여전히 어려움을 겪는 데이터셋과 대규모 웹 코퍼스에서의 훈련 관련 방법론적 문제를 식별하였다. 또한, GPT-3가 생성한 뉴스 기사 샘플이 인간 평가자들이 구별하기 어려운 수준에 이른다는 점을 강조하며, 이러한 발견의 사회적 영향을 논의한다.

## 저자 소개
저자들은 OpenAI 소속의 연구자들로, Tom B. Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared Kaplan 등 다양한 전문가들이 포함되어 있다. 이들은 자연어 처리 및 기계 학습 분야에서의 경험과 전문성을 바탕으로 GPT-3 모델의 개발 및 연구에 기여하였다.


## 실습 : 함수 생성

- input : pdf 파일 경로 
- output : pdf 내용을 요약한 summary.txt 파일 생성 및 summary return

- 파일명: pdf_summary.py